<div style="background-color:#1e1e1e; border: 3px solid #ff00ff; border-radius: 10px; padding: 20px; text-align: center;">
    <h1 style="color:#ff00ff;">Multiclass Cognitive Distortion Classification via PEFT-LoRA</h1>
    <h3 style="color:#ffffff;">Implementing Low-Rank Adaptation on RoBERTa Architecture for Imbalanced Dataset</h3>
</div>

Notebook ini mengimplementasikan teknik **PEFT-LoRA** untuk mengatasi klasifikasi multi-kelas pada dataset yang tidak seimbang.

LoRA bekerja dengan cara menyisipkan matriks kecil ke dalam arsitektur Transformer tanpa mengubah bobot asli model. Teknik ini secara drastis memangkas jumlah parameter yang dilatih, menghemat memori, namun tetap menghasilkan performa maksimal.

### 1. Setup Lingkungan & Import Library

In [ ]:
!pip install transformers datasets evaluate accelerate peft -q
!pip uninstall -y torchao

import pandas as pd
import numpy as np
import torch
import re
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, TrainerCallback
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

warnings.filterwarnings('ignore')

### 2. Data Preprocessing and Label Encoding

In [ ]:
dataset_path = '/kaggle/input/datasets/luthfimuthathohirin/indonesian-cognitive-distortion/COGNITIVE DISTORTION DATASET IN BAHASA INDONESIA COMPLETE.csv'
df = pd.read_csv(dataset_path)

def clean_text_advanced(text):
    text = str(text).replace('$', '')
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    return re.sub(r'\s+', ' ', text).strip()

df = df[df['FIRST ANNOTATOR'] != 'No Distortion'].copy()
df['clean_text'] = df['TEXT'].apply(clean_text_advanced)

le = LabelEncoder()
df['label'] = le.fit_transform(df['FIRST ANNOTATOR'])
NUM_CLASSES = len(le.classes_)

df_ori = df[df['DATA STATUS'] == 'RAW-ORI'].copy()
df_aug = df[df['DATA STATUS'] != 'RAW-ORI'].copy()

train_ori, test_df = train_test_split(df_ori, test_size=0.15, random_state=42, stratify=df_ori['label'])
train_df = pd.concat([train_ori, df_aug], ignore_index=True)
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
train_df = train_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'})
test_df = test_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'})

### 3. Model Initialization and Tokenization

In [ ]:
model_name = "w11wo/indonesian-roberta-base-sentiment-classifier"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def fungsi_tokenisasi(contoh):
    return tokenizer(contoh['text'], padding="max_length", truncation=True, max_length=256)

train_ds = Dataset.from_pandas(train_df).map(fungsi_tokenisasi, batched=True).remove_columns(['text'])
test_ds = Dataset.from_pandas(test_df).map(fungsi_tokenisasi, batched=True).remove_columns(['text'])

### 4. LoRA Adapter Configuration

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

model_lora = get_peft_model(base_model, peft_config)

### 5. Model Training and Optimization

In [ ]:
def compute_acc(pred):
    kategori_prediksi = pred.predictions.argmax(-1)
    f1 = f1_score(pred.label_ids, kategori_prediksi, average='macro')
    acc = accuracy_score(pred.label_ids, kategori_prediksi)
    return {'accuracy': acc, 'macro_f1': f1}

riwayat_train = []

class PantauTrainCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, **kwargs):
        hasil = pelatih.predict(train_ds)
        akurasi = accuracy_score(hasil.label_ids, hasil.predictions.argmax(-1))
        riwayat_train.append({'epoch': state.epoch, 'train_acc': akurasi, 'train_loss': hasil.metrics['test_loss']})

args = TrainingArguments(
    output_dir='./results_lora_level2',
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=1,
    learning_rate=3e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=15,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    report_to="none"
)

pelatih = Trainer(
    model=model_lora,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_acc,
    callbacks=[PantauTrainCallback()]
)

### 6. Execute Training

In [ ]:
pelatih.train()

### 7. Training Convergence Analysis

In [ ]:
riwayat_log = pelatih.state.log_history

val_epochs = []
val_loss_list = []
val_acc_list = []

for entry in riwayat_log:
    if 'eval_loss' in entry and 'eval_accuracy' in entry:
        val_epochs.append(entry['epoch'])
        val_loss_list.append(entry['eval_loss'])
        val_acc_list.append(entry['eval_accuracy'])

train_epochs = [r['epoch'] for r in riwayat_train]
train_loss_list = [r['train_loss'] for r in riwayat_train]
train_acc_list = [r['train_acc'] for r in riwayat_train]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

ax1.plot(train_epochs, train_loss_list, color='#00FFFF', marker='o', linewidth=2, label='Train Loss')
ax1.plot(val_epochs, val_loss_list, color='#FF00FF', marker='s', linewidth=2, label='Val Loss')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training vs Validation Loss (LoRA)', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, linestyle='--', alpha=0.6)

ax2.plot(train_epochs, train_acc_list, color='#00FFFF', marker='o', linewidth=2, label='Train Accuracy')
ax2.plot(val_epochs, val_acc_list, color='#FF00FF', marker='s', linewidth=2, label='Val Accuracy')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Training vs Validation Accuracy (LoRA)', fontsize=14, fontweight='bold')
ax2.legend(loc='lower right')
ax2.grid(True, linestyle='--', alpha=0.6)

fig.tight_layout()
plt.show()

### 8. Final Model Evaluation

In [ ]:
prediksi_val = pelatih.predict(test_ds)
label_val = prediksi_val.label_ids
prediksi_kelas_val = prediksi_val.predictions.argmax(-1)
akurasi_val = accuracy_score(label_val, prediksi_kelas_val)

print(f"Multiclass Accuracy: {akurasi_val*100:.2f}%\n")
print(classification_report(label_val, prediksi_kelas_val, target_names=le.classes_))

### 9. Multiclass Confusion Matrix

In [ ]:
prediksi_train = pelatih.predict(train_ds)
label_train = prediksi_train.label_ids
prediksi_kelas_train = prediksi_train.predictions.argmax(-1)

matriks_train = confusion_matrix(label_train, prediksi_kelas_train)
matriks_val = confusion_matrix(label_val, prediksi_kelas_val)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 9))

sns.heatmap(matriks_train, annot=True, fmt='d', cmap='BuPu', ax=ax1, 
            xticklabels=le.classes_, yticklabels=le.classes_, annot_kws={"size": 10})
ax1.set_title('Confusion Matrix (Training Data)', fontsize=14, fontweight='bold')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax1.set_yticklabels(ax1.get_yticklabels(), rotation=0, fontsize=9)
ax1.set_xlabel('Predicted Label', fontsize=12)
ax1.set_ylabel('True Label', fontsize=12)

sns.heatmap(matriks_val, annot=True, fmt='d', cmap='BuPu', ax=ax2, 
            xticklabels=le.classes_, yticklabels=le.classes_, annot_kws={"size": 10})
ax2.set_title('Confusion Matrix (Validation Data)', fontsize=14, fontweight='bold')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0, fontsize=9)
ax2.set_xlabel('Predicted Label', fontsize=12)
ax2.set_ylabel('True Label', fontsize=12)

plt.tight_layout()
plt.show()

### 10. Adapter Export for Deployment

In [ ]:
pelatih.model.save_pretrained("/content/lora_adapter_level2")
tokenizer.save_pretrained("/content/lora_adapter_level2")

!zip -r /content/lora_siap_deploy.zip /content/lora_adapter_level2 -q